### Datos Stream desde archivos en la nube
1. Leer archivos de la "nube" utilizando API - DataStreamReader
2. Transformar el DataFrame y agregar las siguientes columnas:
    - file path: Ruta del archivo en la nube
    - ingestion date: Current Timestamp
3. Escribir los datos de Stream transformados en Tabla Delta Lake

In [0]:
%fs ls /Volumes/zenviro/raw/operational_data/locations_stream/

#### 1. Leer archivos de la "nube" utilizando API - DataStreamReader

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, TimestampType

location_schema = StructType(fields = [
    StructField('location_id', IntegerType()),
    StructField('location_name', StringType()),
    StructField('address', StringType()),
    StructField('city', StringType()),
    StructField('country', StringType()),
    StructField('created_timestamp', TimestampType())
])

In [0]:
location_df = (
    spark.readStream
    .format("json")
    .schema(location_schema)
    .option("maxFilesPerTrigger", 1)
    .option("cleanSource", "archive")
    .option("sourceArchiveDir", "/Volumes/zenviro/raw/operational_data/locations_stream_history/")
    .load("/Volumes/zenviro/raw/operational_data/locations_stream/")
)

#### 2. Transformar el DataFrame y agregar las siguientes columnas:
- file path: Ruta del archivo en la nube
- ingestion date: Current Timestamp

In [0]:
from pyspark.sql.functions import current_timestamp, col

location_transformed_df = (
    location_df.withColumn("file_path", col("_metadata.file_path"))
               .withColumn("ingestion_date", current_timestamp())
)

#### 3. Escribir los datos de Stream transformados en Tabla Delta Lake

In [0]:
streaming_query = (
    location_transformed_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "/Volumes/zenviro/raw/operational_data/locations_stream/_checkpoint_stream")
    .toTable("zenviro.bronze.locations_stream")
)

In [0]:
%sql
SELECT * FROM zenviro.bronze.locations_stream;